# Risk rating via `lic_dsf.rating`

CI Summary thresholds, Chart Data mechanical ratings, Output 7.
See `docs/10-risk-rating.qmd`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.load import load_core, load_rating
from lic_dsf.rating import (
    ChartDataRegistry,
    RiskRatingSummary,
    compute_mechanical_ratings,
)
from lic_dsf.output import risk_summary_panel

for cand in (Path.cwd(), Path.cwd().parent):
    if (cand / "data" / "lic-dsf-template-2025-08-12.xlsx").exists():
        REPO = cand
        break
WB = REPO / "data" / "lic-dsf-template-2025-08-12.xlsx"

# Edit CI / Ext / Macro in Excel, save, then reload.
macro, external, ext_base, pub_base = load_core(WB)
rating = load_rating(WB)
print(rating.ci.country, rating.ci.dcc, rating.ci.ci_score)
print(rating.ci.thresholds)

ext_b = ext_base
pub_b = pub_base

years = [y for y in ext_b.years if y >= macro.inputs.first_projection_year][:11]
registry = ChartDataRegistry()
registry.register_series(
    "pv_debt_to_gdp", "baseline", ext_b.pv_ppg_external_to_gdp().reindex(years), is_baseline=True
)
registry.register_series(
    "pv_debt_to_exports", "baseline", ext_b.pv_ppg_external_to_exports().reindex(years), is_baseline=True
)
registry.register_series(
    "debt_service_to_exports", "baseline", ext_b.ppg_debt_service_to_exports().reindex(years), is_baseline=True
)
registry.register_series(
    "debt_service_to_revenue", "baseline", ext_b.ppg_debt_service_to_revenue().reindex(years), is_baseline=True
)
registry.register_series(
    "public_pv_debt_to_gdp", "baseline", pub_b.pv_public_debt_to_gdp().reindex(years), is_baseline=True
)

mech = compute_mechanical_ratings(registry, rating.ci.thresholds, years=years)
summary = RiskRatingSummary(
    mechanical=mech, thresholds=rating.ci.thresholds, dcc=rating.ci.dcc, ci_score=rating.ci.ci_score
)
risk_summary_panel(summary)

